# Fake Review Detection Model
Training SVM and Logistic Regression models to detect fake reviews

In [ ]:
# Install required libraries
!pip install scikit-learn pandas numpy nltk joblib -q

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

In [ ]:
# Load or create sample training data
# In production, load your actual dataset

# Create sample data for demonstration
data = {
    'review': [
        'Great product, excellent quality and fast shipping!',
        'Best purchase ever, highly recommend!',
        'Amazing item, works perfectly',
        'Love it! Five stars',
        'This is fake, do not trust',
        'Paid review, not genuine opinion',
        'Buy now! Limited offer!!!',
        'Click here for discount',
    ],
    'label': [0, 0, 0, 0, 1, 1, 1, 1]  # 0=Genuine, 1=Fake
}

df = pd.DataFrame(data)
print('Dataset shape:', df.shape)
print('\nLabel distribution:')
print(df['label'].value_counts())

In [ ]:
# Text Preprocessing
from nltk.stem import PorterStemmer
import re

stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove special characters
    text = re.sub(r'[^a-z0-9\s]', '', text)
    
    # Tokenize and remove stopwords
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(token) for token in tokens if token not in stop_words]
    
    return ' '.join(tokens)

df['review_processed'] = df['review'].apply(preprocess_text)
print('Preprocessing complete')
print('\nSample processed review:')
print(df['review_processed'].iloc[0])

In [ ]:
# Feature Extraction - TF-IDF Vectorization

vectorizer = TfidfVectorizer(max_features=5000, min_df=1, max_df=0.95)
X = vectorizer.fit_transform(df['review_processed'])
y = df['label']

print('Feature matrix shape:', X.shape)
print('Number of features:', X.shape[1])
print('\nVocabulary size:', len(vectorizer.get_feature_names_out()))

In [ ]:
# Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training set size:', X_train.shape[0])
print('Testing set size:', X_test.shape[0])

In [ ]:
# Train SVM Model (Primary)

print('Training SVM Classifier...')
svm_model = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
svm_model.fit(X_train, y_train)

# Predictions
y_pred_svm = svm_model.predict(X_test)

# Evaluation
print('\n=== SVM Model Performance ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_svm):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_svm):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred_svm):.4f}')

In [ ]:
# Train Logistic Regression Model (Comparison)

print('Training Logistic Regression Classifier...')
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test)

# Evaluation
print('\n=== Logistic Regression Performance ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_lr):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_lr):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred_lr):.4f}')

In [ ]:
# Train Random Forest Model (Optional)

print('Training Random Forest Classifier...')
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)

# Evaluation
print('\n=== Random Forest Performance ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_rf):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_rf):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred_rf):.4f}')

In [ ]:
# Model Comparison

print('\n=== Model Comparison ===')

models_comparison = {
    'SVM': accuracy_score(y_test, y_pred_svm),
    'Logistic Regression': accuracy_score(y_test, y_pred_lr),
    'Random Forest': accuracy_score(y_test, y_pred_rf)
}

for model_name, accuracy in sorted(models_comparison.items(), key=lambda x: x[1], reverse=True):
    print(f'{model_name}: {accuracy:.4f}')

In [ ]:
# Save models to disk

import os

# Create directory if it doesn't exist
os.makedirs('/content/models', exist_ok=True)

# Save best model (SVM)
joblib.dump(svm_model, '/content/models/fake_review_model.pkl')
joblib.dump(vectorizer, '/content/models/vectorizer.pkl')

print('✓ SVM model saved')
print('✓ Vectorizer saved')

# Also save other models for comparison
joblib.dump(lr_model, '/content/models/fake_review_lr.pkl')
joblib.dump(rf_model, '/content/models/fake_review_rf.pkl')

print('✓ Logistic Regression model saved')
print('✓ Random Forest model saved')

In [ ]:
# Test predictions with new data

test_reviews = [
    'This product is amazing!',
    'Click here to buy now!!!',
    'Best quality ever',
]

print('=== Test Predictions ===\n')

for review in test_reviews:
    processed = preprocess_text(review)
    X_review = vectorizer.transform([processed])
    
    prediction = svm_model.predict(X_review)[0]
    probability = svm_model.predict_proba(X_review)[0]
    
    label = 'Fake' if prediction == 1 else 'Genuine'
    confidence = max(probability)
    
    print(f'Review: {review}')
    print(f'Prediction: {label} (Confidence: {confidence:.2%})\n')